In [ ]:
import pandas as pd
from energy_model.evaluation.model_evaluator import ModelEvaluator
from sklearn.model_selection import train_test_split
from energy_model.configs.columns import ProcessColumns
from energy_model.pipelines.pipeline_utils import extract_x_y
from sklearn.ensemble import HistGradientBoostingRegressor
from energy_model.configs.columns import SystemColumns


In [ ]:

process_real_time_df = pd.read_csv(r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_process_df_new.csv")

system_real_time_df = pd.read_csv(r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_only_df_new.csv")

process_long_term_df = pd.read_csv(r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\aggregated - system based - process of interest\system_process_df.csv")

system_long_term_df = pd.read_csv(r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\aggregated - system based - process of interest\system_only_df.csv")

In [ ]:
PROCESS_REAL_TIME_MODEL = HistGradientBoostingRegressor
PROCESS_REAL_TIME_MODEL_PARAMETERS = {"max_depth": 8,
                                      "l2_regularization": 0.3,
                                      "loss": "quantile",
                                      "max_iter": 400,
                                      "quantile": 0.5}

PROCESS_LONG_TERM_MODEL = HistGradientBoostingRegressor
PROCESS_LONG_TERM_MODEL_PARAMETERS = {"max_depth": 8,
                                      "l2_regularization": 1.0,
                                      "loss": "quantile",
                                      "max_iter": 600,
                                      "quantile": 0.7}

SYSTEM_REAL_TIME_MODEL = HistGradientBoostingRegressor
SYSTEM_REAL_TIME_MODEL_PARAMETERS = {"max_depth": 8,
                                     "l2_regularization": 1.0,
                                     "loss": "quantile",
                                     "max_iter": 600,
                                     "quantile": 0.7}

SYSTEM_LONG_TERM_MODEL = HistGradientBoostingRegressor
SYSTEM_LONG_TERM_MODEL_PARAMETERS = {"max_depth": 8,
                                     "l2_regularization": 1.0,
                                     "loss": "quantile",
                                     "max_iter": 600,
                                     "quantile": 0.7}

In [ ]:
durations = [5, 10, 15]

In [ ]:
def get_durations_performance(df: pd.DataFrame, model_type, model_parameters):
    previous_duration_threshold = 0
    for current_duration_threshold in durations:
        print(f"Calculating for {previous_duration_threshold} < duration <= {current_duration_threshold}")
        duration_based_df = df[(df[SystemColumns.DURATION_COL] > previous_duration_threshold) &
                               (df[SystemColumns.DURATION_COL] <= current_duration_threshold)]
        print(f"df shape: {duration_based_df.shape}")
        previous_duration_threshold = current_duration_threshold
        model = model_type(**model_parameters)
        print("built model")
        x, y = extract_x_y(duration_based_df, target_column=ProcessColumns.ENERGY_USAGE_PROCESS_COL)
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
        print("Starting fit")
        model.fit(x_train, y_train)
        print("Finished fit, calling predict")
        y_pred = model.predict(x_test)
        ev = ModelEvaluator()
        print("Evaluating..")
        res = ev.evaluate(y_test, y_pred)
        ev.print_results(res)



In [ ]:
def get_durations_performance_sys(df: pd.DataFrame, model_type, model_parameters):
    previous_duration_threshold = 0
    for current_duration_threshold in durations:
        print(f"Calculating for {previous_duration_threshold} < duration <= {current_duration_threshold}")
        duration_based_df = df[(df[SystemColumns.DURATION_COL] > previous_duration_threshold) &
                               (df[SystemColumns.DURATION_COL] <= current_duration_threshold)]
        print(f"df shape: {duration_based_df.shape}")
        previous_duration_threshold = current_duration_threshold
        model = model_type(**model_parameters)
        print("built model")
        x, y = extract_x_y(duration_based_df, target_column=SystemColumns.ENERGY_USAGE_SYSTEM_COL)
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
        print("Starting fit")
        model.fit(x_train, y_train)
        print("Finished fit, calling predict")
        y_pred = model.predict(x_test)
        ev = ModelEvaluator()
        print("Evaluating..")
        res = ev.evaluate(y_test, y_pred)
        ev.print_results(res)



# Real Time Process Model

In [ ]:
get_durations_performance(process_real_time_df, PROCESS_REAL_TIME_MODEL, PROCESS_REAL_TIME_MODEL_PARAMETERS)

# Real Time System Model

In [ ]:
get_durations_performance_sys(system_real_time_df, SYSTEM_REAL_TIME_MODEL, SYSTEM_REAL_TIME_MODEL_PARAMETERS)

# Long Term Process Model

In [ ]:
get_durations_performance(process_long_term_df, PROCESS_LONG_TERM_MODEL, PROCESS_LONG_TERM_MODEL_PARAMETERS)

# Long Term System Model

In [ ]:
get_durations_performance(system_long_term_df, SYSTEM_LONG_TERM_MODEL, SYSTEM_LONG_TERM_MODEL_PARAMETERS)